# Old -> New Annotation Conversion Rule

**Update:** the new-version ground truth files were updated and renamed since this rule was first derived (inconsistently — see the file-resolution cell below). Re-validating against the current files.

**The generalized rule:**

For every question in a document, in one pass:

1. If `question.text` is already non-empty, leave it unchanged.
2. If `question.text` is empty:
   - If its own section has a non-empty `title`, set `question.text = section.title`.
   - Otherwise (section title also empty), set `question.text = document.title` (only the first time this happens), and clear `document.title = ""`.

No section-collapsing, no per-sample logic, no sample numbers anywhere in the rule — it only ever looks at whether a given piece of text is empty and what the nearest non-empty title is.

This notebook is exploratory only — nothing here is wired into the package/pipeline.

In [1]:
import copy
import json
import re
from pathlib import Path

from dmpbridge.evaluation.evaluate import LLM_DIR, MANUAL_DIR, NEW_MANUAL_DIR

## Resolve new-version ground truth filenames

The new-version files no longer follow one consistent naming pattern:

- samples 1, 2, 3, 4, 7 &rarr; `sampleN_dmp_new.json`
- samples 5, 6, 8, 9, 10 &rarr; `dmp_sampleN_new.json`

This scans the directory by sample number instead of hardcoding either pattern, so it keeps working regardless of which naming convention a given file uses.

In [2]:
def new_gt_path(n: int) -> Path:
    """Resolve sample n's new-version ground truth file, regardless of naming pattern."""
    for p in NEW_MANUAL_DIR.glob('*.json'):
        m = re.fullmatch(r'\D*(\d+)\D*', p.stem)
        if m and int(m.group(1)) == n:
            return p
    raise FileNotFoundError(f'No new-version ground truth file found for sample{n} in {NEW_MANUAL_DIR}')


NEW_GT_FILES = {n: new_gt_path(n) for n in range(1, 11)}
for n, p in NEW_GT_FILES.items():
    print(f'sample{n:<2} -> {p.name}')

sample1  -> sample1_dmp_new.json
sample2  -> sample2_dmp_new.json
sample3  -> sample3_dmp_new.json
sample4  -> sample4_dmp_new.json
sample5  -> dmp_sample5_new.json
sample6  -> dmp_sample6_new.json
sample7  -> sample7_dmp_new.json
sample8  -> dmp_sample8_new.json
sample9  -> dmp_sample9_new.json
sample10 -> dmp_sample10_new.json


## The conversion function

In [3]:
def apply_new_annotation_rules(data: dict) -> dict:
    """Return a copy of *data* with Rules 1 and 3 applied (see notebook intro)."""
    data     = copy.deepcopy(data)
    root     = data.get('narrative', data)
    template = root.get('template', {})
    doc_title  = template.get('title', '').strip()
    title_used = False

    for section in template.get('section', []):
        sec_title = section.get('title', '').strip()
        for question in section.get('question', []):
            if question.get('text', '').strip():
                continue
            if sec_title:
                question['text'] = sec_title
            elif doc_title and not title_used:
                question['text'] = doc_title
                title_used = True

    if title_used:
        template['title'] = ''

    return data

## Validate against real ground truth

Compares the converted output against the current new-version file for every sample. Reports both a raw exact match and a "content match" that ignores the `template.version` field ("V1" vs "v1" — a metadata capitalization difference with nothing to do with the annotation content).

In [4]:
def without_version(structured: dict) -> dict:
    d = copy.deepcopy(structured)
    d['narrative']['template'].pop('version', None)
    return d


for n in range(1, 11):
    old = json.loads((MANUAL_DIR / f'sample{n}_old_dmp.json').read_text(encoding='utf-8'))
    new = json.loads(NEW_GT_FILES[n].read_text(encoding='utf-8'))
    got = apply_new_annotation_rules(old)

    raw_match     = got == new
    content_match = without_version(got) == without_version(new)
    print(f'sample{n:<2}  raw match: {str(raw_match):5s}  content match (ignoring version field): {content_match}')

sample1   raw match: True   content match (ignoring version field): True
sample2   raw match: False  content match (ignoring version field): False
sample3   raw match: True   content match (ignoring version field): True
sample4   raw match: True   content match (ignoring version field): True
sample5   raw match: False  content match (ignoring version field): False
sample6   raw match: False  content match (ignoring version field): True
sample7   raw match: True   content match (ignoring version field): True
sample8   raw match: False  content match (ignoring version field): True
sample9   raw match: False  content match (ignoring version field): True
sample10  raw match: False  content match (ignoring version field): True


## Remaining mismatches, characterized

After the ground truth update, section-collapsing is gone — every sample's section count now matches the rule's output. Only 2 of 10 samples still differ in content:

- **sample2**: 3 tiny, unrelated text edits in the new file (a missing quotation mark added, two double-spaces fixed to single-spaces). Nothing to do with the rule.
- **sample5**: a genuinely new pattern — 2 of its 6 sections had multiple old sub-questions (e.g. "Raw data:", "Scripts and code for analyses:") that got **merged into one question** in the new file, with the sub-labels folded into the start of the merged answer text. The rule doesn't do this (it only fills empty question text; it never merges non-empty ones).

In [5]:
old = json.loads((MANUAL_DIR / 'sample5_old_dmp.json').read_text(encoding='utf-8'))
new = json.loads(NEW_GT_FILES[5].read_text(encoding='utf-8'))
got = apply_new_annotation_rules(old)

for gs, ns in zip(got['narrative']['template']['section'], new['narrative']['template']['section']):
    flag = '  <- merged' if len(gs['question']) != len(ns['question']) else ''
    print(f"{gs['title'][:40]:<40}  rule={len(gs['question'])} question(s)  "
          f"new-gt={len(ns['question'])} question(s){flag}")

REVIEW OF PROPOSAL COMPONENTS             rule=1 question(s)  new-gt=1 question(s)
TYPES OF DATA                             rule=3 question(s)  new-gt=1 question(s)  <- merged
DATA AND METADATA STANDARDS               rule=1 question(s)  new-gt=1 question(s)
POLICIES FOR ACCESS AND SHARING           rule=4 question(s)  new-gt=1 question(s)  <- merged
POLICIES AND PROVISIONS FOR RE-USE, RE-D  rule=1 question(s)  new-gt=1 question(s)
PLANS FOR ARCHIVING AND PRESERVATION OF   rule=1 question(s)  new-gt=1 question(s)


## Convert & save: old ground truth -> new format (all 10 samples)

Applies the rules to every `sampleN_old_dmp.json` and saves the result to `data/output/ground_truth_converted_test/sampleN_dmp.json`, so you can open any of them directly. Also reports, per sample, whether it matches the real new-version annotation exactly (expected `True` for 1, 2, 3, 4, 7 and `False` for the 5 collapse-case samples, per the validation above).

In [6]:
GT_OUTPUT_DIR = LLM_DIR.parent / 'ground_truth_converted_test'
GT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for n in range(1, 11):
    old = json.loads((MANUAL_DIR / f'sample{n}_old_dmp.json').read_text(encoding='utf-8'))
    new = json.loads(NEW_GT_FILES[n].read_text(encoding='utf-8'))
    converted = apply_new_annotation_rules(old)

    out_path = GT_OUTPUT_DIR / f'sample{n}_dmp.json'
    out_path.write_text(json.dumps(converted, indent=2, ensure_ascii=False), encoding='utf-8')

    raw_match     = converted == new
    content_match = without_version(converted) == without_version(new)
    print(f'sample{n:<2}  raw match: {str(raw_match):5s}  content match: {str(content_match):5s}  ->  {out_path.name}')

print(f'\nSaved 10 files under: {GT_OUTPUT_DIR}')

sample1   raw match: True   content match: True   ->  sample1_dmp.json
sample2   raw match: False  content match: False  ->  sample2_dmp.json
sample3   raw match: True   content match: True   ->  sample3_dmp.json
sample4   raw match: True   content match: True   ->  sample4_dmp.json
sample5   raw match: False  content match: False  ->  sample5_dmp.json
sample6   raw match: False  content match: True   ->  sample6_dmp.json
sample7   raw match: True   content match: True   ->  sample7_dmp.json
sample8   raw match: False  content match: True   ->  sample8_dmp.json
sample9   raw match: False  content match: True   ->  sample9_dmp.json
sample10  raw match: False  content match: True   ->  sample10_dmp.json

Saved 10 files under: C:\Users\Nahid\dmpbridge\data\output\ground_truth_converted_test
